Exp 6

Amishi Gupta
23/CS/048

In [1]:
!pip install torchtext --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 26.8 MB/s eta 0:00:00


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import pandas as pd

file_path = "/content/drive/MyDrive/imdb_dataset.csv"

df = pd.read_csv(file_path)

print(df.head())
print(df.columns)

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
Index(['review', 'sentiment'], dtype='object')


In [11]:
df = df.rename(columns={"review": "text", "sentiment": "label"})

In [12]:
df["label"] = df["label"].map({"positive": 1, "negative": 0})

In [7]:
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)
    return text

df["text"] = df["text"].apply(clean_text)

In [8]:
#tokenisation
tokens = []
for sentence in df["text"]:
    tokens.extend(sentence.split())
vocab = sorted(set(tokens))
vocab_size = len(vocab)
word2idx = {w:i+1 for i,w in enumerate(vocab)}

In [9]:
#encode
def encode(sentence):
    return [word2idx.get(w, 0) for w in sentence.split()]
df["encoded"] = df["text"].apply(encode)

In [13]:
import torch
max_len = 100
def pad(seq):
    if len(seq) < max_len:
        seq += [0]*(max_len - len(seq))
    else:
        seq = seq[:max_len]
    return seq
df["padded"] = df["encoded"].apply(pad)
X = torch.tensor(df["padded"].tolist())
y = torch.tensor(df["label"].tolist())

In [14]:
import torch.nn as nn
class RNNModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden_size=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size+1, embed_dim)
        self.rnn = nn.RNN(embed_dim, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.rnn(x)
        out = self.fc(out[:, -1, :])
        return self.sigmoid(out)

In [15]:
#train
import torch.optim as optim
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = RNNModel(vocab_size).to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [16]:
batch_size = 64
epochs = 5

for epoch in range(epochs):
    total_loss = 0
    model.train()

    for i in range(0, len(X), batch_size):
        inputs = X[i:i+batch_size].to(device)
        labels = y[i:i+batch_size].float().to(device)
        optimizer.zero_grad()
        outputs = model(inputs).squeeze()
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

Epoch 1, Loss: 544.2430
Epoch 2, Loss: 539.7176
Epoch 3, Loss: 540.4799
Epoch 4, Loss: 535.2258
Epoch 5, Loss: 518.2071


In [17]:
#accuracy
correct = 0
total = 0
model.eval()
with torch.no_grad():
    outputs = model(X.to(device)).squeeze()
    preds = (outputs > 0.5).int().cpu()

    correct = (preds == y).sum().item()
    total = len(y)
print("Accuracy:", 100 * correct / total, "%")

Accuracy: 64.846 %
